# Lekcia 05 - Agentický RAG


## Nastavenie

Tento poznámkový blok demonštruje vzor Agentic RAG (Retrieval-Augmented Generation) pomocou Microsoft Agent Framework.

**Predpoklady:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — váš koncový bod služby Azure AI Search
- `AZURE_SEARCH_API_KEY` — váš API kľúč pre Azure AI Search
- Azure OpenAI nasadenie nakonfigurované prostredníctvom premenných prostredia
- Azure CLI autentifikovaný (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Čo je Agentic RAG?

Tradičný RAG nasleduje pevnú postupnosť: najprv vyhľadá dokumenty, potom vygeneruje odpoveď. **Agentic RAG** ide ďalej tým, že dáva agentovi autonómiu rozhodovať **kedy** a **ako** získať informácie.

S Agentic RAG môže agent:
- **Rozhodnúť** sa, či je potrebné vyhľadávanie pred odpoveďou na otázku
- **Vybrať** ktorý zdroj dát alebo nástroj použiť na vyhľadávanie
- **Vyhodnotiť** získané výsledky a vykonať ďalšie vyhľadávania, ak prvý pokus nestačí
- **Kombinovať** informácie z viacerých krokov vyhľadávania do zrozumiteľnej odpovede

Toto robí agenta flexibilnejším a presnejším v porovnaní so statickým modelom najprv vyhľadať-potom-vygenerovať.


## Vytváranie vyhľadávacieho nástroja

V Agentic RAG sú externé zdroje dát zabalené ako **nástroje**, ktoré môže agent na požiadanie vyvolať. To umožňuje agentovi považovať získavanie informácií len za ďalšiu akciu, ktorú môže vykonať, namiesto povinného kroku.

Nižšie definujeme základňu znalostí o cestovaní a sprístupníme ju ako nástroj, ktorý môže agent použiť na vyhľadávanie informácií o destináciách.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Vytváranie RAG agenta

Teraz vytvoríme agenta, ktorý je inštruovaný, aby **vždy pred odpoveďou najprv vyhľadával informácie**. Agent používa nástroj `search_travel_knowledge` na to, aby svoje odpovede zakladal na znalostnej báze namiesto toho, aby sa spoliehal na svoje vlastné tréningové dáta.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iteratívne získavanie — Vzor tvorca-kontrolór

Kľúčovou výhodou Agentic RAG je **iteratívne získavanie**. Agent môže vykonať niekoľko kôl vyhľadávania na overenie, doplnenie alebo rozšírenie svojich počiatočných zistení — podobne ako pracovný postup „tvorca-kontrolór“:

1. **Krok tvorcu**: Agent získava počiatočné informácie a zostavuje odpoveď.
2. **Krok kontrolóra**: Agent vykonáva ďalšie vyhľadávania na overenie detailov alebo zaplnenie medzier.

Nižšie je agentovi položená otázka, ktorá vyžaduje porovnávanie viacerých destinácií, čo ho podnecuje vyhľadávať viackrát.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Súhrn

V tejto lekcii ste sa naučili, ako vytvoriť systém **Agentic RAG** pomocou Microsoft Agent Frameworku:

- **Agentic RAG** umožňuje agentom autonómne rozhodovať, kedy získavať informácie, čím sa získavanie stáva dynamickým, nie fixným.
- **Nástroje ako zdroje dát**: Externé znalostné bázy (ako Azure AI Search) sú zabalené ako nástroje, ktoré môže agent vyvolať.
- **Iteratívne získavanie**: Vzor maker-checker umožňuje agentovi vykonať viacero kôl získavania — hľadanie, overovanie a zdokonaľovanie — pred vytvorením konečnej odpovede.

V produkcii by ste nahradili pamäťovú `TRAVEL_KNOWLEDGE_BASE` skutočným indexom Azure AI Search na spracovanie veľkorozmerného získavania dokumentov o cestovaní.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
